# 🕵️ Case file 02 — The answer the booking system couldn't read

> **Mission briefing.** Same chaotic day, new incident. The downstream booking system rejected an answer from the assistant because it was not machine-readable. The assistant had been asked to summarize a booking request in plain English. Your job is to compare that answer with a constrained one and work out which one the system can actually use.

The next system is software, not a person: it needs exact fields with the right types before it can book anything.

## Case objective
Compare a free-text answer with a structured answer, and check whether each one has the fields the next system needs.

## What to expect
The free-text run produces prose that fails validation. The structured run returns exactly the required fields and passes. The activity logs are `activity-2-output-unconstrained` and `activity-2-output-constrained`.

## Bad practice
Ask for a free-text answer and try to pull out the fields with pattern matching.

## Good practice
Ask for a structured answer with the exact fields you need, then check those fields before using them.

## Why the good practice works
A structured answer is easy to check. If a field is missing or has the wrong type, validation fails with a clear reason instead of passing bad data to the next system.

## Deliverable
The two activity-log links and the result of checking each answer.


In [1]:
from IPython.display import Markdown, display
from experiment_utils import RUN_ID, langfuse, run_format


## 🔮 Your prediction

The booking system expects exactly `origin`, `destination`, `travel_date`, `passengers`, and `cabin`, with strict types.

**Will the free-text answer pass that check?** What about the structured one?

Write your guess below (double-click this cell) or as a comment in the next cell.

---
*Your guess:*


In [2]:
runs = [run_format("unconstrained"), run_format("constrained")]

for run in runs:
    if run["valid"]:
        print(f"✅  {run['label'].capitalize()} answer accepted by the downstream system.")
    else:
        print(f"⚠️  {run['label'].capitalize()} answer rejected: {run['reason']}.")

rows = ["| run | accepted? | reason | answer | input tokens | output tokens | time (s) | activity log |",
        "|---|---|---|---|---|---|---|---|"]
for run in runs:
    accepted = "yes" if run["valid"] else "**no**"
    reason = (run["reason"] or "").replace("|", "\\|")
    answer = (run["answer"] or "").replace("\n", " ").replace("|", "\\|")
    rows.append(f"| {run['label']} | {accepted} | {reason} | {answer} | {run['input_tokens']} | {run['output_tokens']} | {run['latency']:.1f} | [Open log]({run['trace_url']}) |")
display(Markdown("\n".join(rows)))

report = [f"- **{run['label']}:** {'accepted' if run['valid'] else 'rejected'} — {run['reason']}" for run in runs]
display(Markdown("**Validation report**\n\n" + "\n".join(report)))


⚠️  Free text answer rejected: the answer is not a structured object.
✅  Structured answer accepted by the downstream system.


| run | accepted? | reason | answer | input tokens | output tokens | time (s) | activity log |
|---|---|---|---|---|---|---|---|
| free text | **no** | the answer is not a structured object | The customer requests a booking for 2 passengers in economy class from Lisbon to Madrid on December 1, 2026. | 60 | 66 | 1.2 | [Open log](http://localhost:3000/project/langfuse-observability/traces/431e305b6abf955a1a014c1e4904e3f1) |
| structured | yes | all fields present and correct | {"origin":"Lisbon","destination":"Madrid","travel_date":"2026-12-01","passengers":2,"cabin":"economy"} | 102 | 65 | 1.0 | [Open log](http://localhost:3000/project/langfuse-observability/traces/9b3b1ad34b1566245fdbfd1937cfd49d) |

**Validation report**

- **free text:** rejected — the answer is not a structured object
- **structured:** accepted — all fields present and correct

## 🔍 Reading the evidence

Open each activity log and compare the generation's system prompt. The unconstrained run asks for one sentence; the constrained run lists the exact keys and types. The trace also records the validation result, so you can see why the first answer never reaches the booking system.

The lesson is not "JSON is nicer". It is that a checkable contract lets the system reject bad data before it causes damage.


## 🏁 Wrap-up

Run the final cell to close the case, show the activity-log links, and flush the traces to Langfuse. Submit the links and your findings.


In [3]:
lines = [f"**Session:** `activity-2-{RUN_ID}`", "", "**Activity logs:**", ""]
for run in runs:
    lines.append(f"- [{run['label']}]({run['trace_url']})")
display(Markdown("\n".join(lines)))
langfuse.flush()
print("✅ Incident closed: the structured answer passed every field check.")
print("🕵️ Detective score: 2/4 incidents resolved. Next case: the two-bag hallucination (Activity 3).")
display(Markdown("Activity logs sent to Langfuse. Open them and compare the system prompts."))


**Session:** `activity-2-422a4ae6`

**Activity logs:**

- [free text](http://localhost:3000/project/langfuse-observability/traces/431e305b6abf955a1a014c1e4904e3f1)
- [structured](http://localhost:3000/project/langfuse-observability/traces/9b3b1ad34b1566245fdbfd1937cfd49d)

✅ Incident closed: the structured answer passed every field check.
🕵️ Detective score: 2/4 incidents resolved. Next case: the two-bag hallucination (Activity 3).


Activity logs sent to Langfuse. Open them and compare the system prompts.

## 🎁 Bonus — What else would you try?

- Add an enum constraint for `cabin` and check whether the model still complies.
- Try a deliberately malformed request and see which validation error fires first.
- Compare token counts: is the structured answer cheaper or more expensive to produce?
